# 复现 Fig. 8：两体直接捕获的总共动并合率

这个 Notebook 复现论文式 (18) 和 Fig. 8。主体物理仍保存在 `twobodycapture.py` 中，本文件只负责：

1. 设置红移网格与两种 PBH 质量函数；
2. 分别调用 Ludlow16 和 Prada12 浓度模型；
3. 用 Press-Schechter 晕质量函数对每晕率积分；
4. 导出 Fig. 8 的 PNG 和 CSV。

论文式 (18) 为

$$
R(z)=\int R_{\rm halo}(M,z)\,\frac{dn}{dM}\,dM.
$$

程序实际使用完全等价、数值上更稳定的对数质量形式

$$
R(z)=\int R_{\rm halo}(M,z)\,\frac{dn}{d\ln M}\,d\ln M.
$$

## 1. 导入模块并定位项目目录

如果从项目根目录运行，`Path.cwd()` 就是项目目录；如果从 `notebooks` 目录运行，则自动退回上一级。

In [ ]:
from pathlib import Path
import sys
import time

import matplotlib.pyplot as plt
import numpy as np

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "twobodycapture.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pbhmassfunction as pmf
import twobodycapture as capture

PROJECT_ROOT

## 2. Fig. 8 的公共参数

论文 Fig. 8 使用：

- $0\le z\le12$；
- Press-Schechter 晕质量函数；
- 晕质量积分范围 $10^3$ 到 $10^{15}M_\odot$；
- 单色 PBH 质量 $m=30M_\odot$；
- 对数正态质量函数 $M_c=30M_\odot$、$\sigma=0.6$；
- Ludlow16 和 Prada12 两种浓度模型。

这里取 41 个线性红移点。晕质量网格使用 0.1 dex；`twobodycapture.py` 会把 HMFcalc 后端 `hmf` 的 $M_\odot/h$ 和 $h^3\,\mathrm{Mpc}^{-3}$ 转成物理单位。

In [ ]:
REDSHIFT = np.linspace(0.0, 12.0, 41)
MINIMUM_HALO_MASS_MSUN = 1.0e3
MAXIMUM_HALO_MASS_MSUN = 1.0e15
HALO_MASS_STEP_DEX = 0.1
F_PBH = 1.0

monochromatic_nodes, monochromatic_weights = (
    pmf.monochromatic_mass_distribution(mass_msun=30.0)
)

def figure8_lognormal_pdf(mass_msun):
    return pmf.lognormal_mass_pdf(
        mass_msun,
        median_mass_msun=30.0,
        sigma=0.6,
    )

lognormal_nodes, lognormal_weights = pmf.continuous_mass_quadrature(
    figure8_lognormal_pdf,
    point_count=96,
)

print(f"红移点数：{REDSHIFT.size}")
print(f"单色谱权重和：{monochromatic_weights.sum():.12f}")
print(f"对数正态谱权重和：{lognormal_weights.sum():.12f}")

## 3. 计算式 (18) 的四条曲线

每次调用 `comoving_capture_rate_eq18_gpc3_per_year` 都完成以下链条：

$$
(M,z)\rightarrow C(M,z)\rightarrow R_{\rm halo}(M,z)\rightarrow\frac{dn}{d\ln M}\rightarrow R(z).
$$

这里 HMF 质量网格上的 $M$ 是红移 $z$ 时实际存在的晕质量，不是今天的质量 $M_0$，因此不能再调用一次质量吸积史。计算约需一分钟；进度信息只用于观察 Notebook 是否仍在运行。

In [ ]:
def calculate_figure8_curve(
    concentration_model,
    mass_nodes_msun,
    mass_probability_weights,
    curve_label,
):
    rate_values = []

    for index, z in enumerate(REDSHIFT, start=1):
        rate = capture.comoving_capture_rate_eq18_gpc3_per_year(
            z=z,
            mass_nodes_msun=mass_nodes_msun,
            mass_probability_weights=mass_probability_weights,
            concentration_model=concentration_model,
            f_pbh=F_PBH,
            minimum_halo_mass_msun=MINIMUM_HALO_MASS_MSUN,
            maximum_halo_mass_msun=MAXIMUM_HALO_MASS_MSUN,
            dlog10m=HALO_MASS_STEP_DEX,
        )
        rate_values.append(rate)

        if index == 1 or index % 10 == 0 or index == REDSHIFT.size:
            print(
                f"{curve_label}: {index:2d}/{REDSHIFT.size}, "
                f"z={z:4.1f}, R={rate:10.4e} Gpc^-3 yr^-1"
            )

    return np.asarray(rate_values)

start_time = time.perf_counter()

fig8_mono_ludlow = calculate_figure8_curve(
    "ludlow16",
    monochromatic_nodes,
    monochromatic_weights,
    "mono.+Ludlow16",
)
fig8_lognormal_ludlow = calculate_figure8_curve(
    "ludlow16",
    lognormal_nodes,
    lognormal_weights,
    "log-normal+Ludlow16",
)
fig8_mono_prada = calculate_figure8_curve(
    "prada12",
    monochromatic_nodes,
    monochromatic_weights,
    "mono.+Prada12",
)
fig8_lognormal_prada = calculate_figure8_curve(
    "prada12",
    lognormal_nodes,
    lognormal_weights,
    "log-normal+Prada12",
)

elapsed_seconds = time.perf_counter() - start_time
print(f"四条曲线计算完成，用时 {elapsed_seconds:.1f} s。")

## 4. 查看端点数值

先显示 $z=0$ 和 $z=12$ 的实际结果。这不是自动验收，而是让我们在看图前知道纵轴数值来自哪里。

In [ ]:
for label, values in [
    ("mono.+Ludlow16", fig8_mono_ludlow),
    ("log-normal+Ludlow16", fig8_lognormal_ludlow),
    ("mono.+Prada12", fig8_mono_prada),
    ("log-normal+Prada12", fig8_lognormal_prada),
]:
    print(
        f"{label:24s}  "
        f"R(0)={values[0]:10.4e},  R(12)={values[-1]:10.4e}"
    )

## 5. 绘制并保存 Fig. 8

颜色、线型、线性坐标范围和图例顺序按照论文原图设置。纵轴保留论文约 $0$ 到 $160\,\mathrm{Gpc}^{-3}\,\mathrm{yr}^{-1}$ 的范围，避免自动缩放掩盖当前计算与原图的幅度差异。

In [ ]:
figure8, axis8 = plt.subplots(figsize=(7.0, 5.2))

axis8.plot(
    REDSHIFT, fig8_mono_ludlow,
    color="tab:blue", linestyle="-", linewidth=1.8,
    label="mono.+Ludlow16",
)
axis8.plot(
    REDSHIFT, fig8_lognormal_ludlow,
    color="tab:orange", linestyle="--", linewidth=1.8,
    label="log-normal+Ludlow16",
)
axis8.plot(
    REDSHIFT, fig8_mono_prada,
    color="tab:green", linestyle="-.", linewidth=1.8,
    label="mono.+Prada12",
)
axis8.plot(
    REDSHIFT, fig8_lognormal_prada,
    color="tab:red", linestyle=":", linewidth=1.8,
    label="log-normal+Prada12",
)

axis8.set_xlim(0.0, 12.0)
axis8.set_ylim(0.0, 165.0)
axis8.set_xticks(np.arange(0.0, 12.1, 2.0))
axis8.set_yticks([0.0, 50.0, 100.0, 150.0])
axis8.set_xlabel(r"Redshift $z$")
axis8.set_ylabel(
    r"$R_{\rm total}\;(\mathrm{Gpc}^{-3}\,\mathrm{yr}^{-1})$"
)
axis8.legend(loc="upper left", fontsize=9.0)
axis8.tick_params(direction="in", top=True, right=True)

figure8.tight_layout()
figure8_path = PROJECT_ROOT / "fig8_reproduction.png"
figure8.savefig(figure8_path, dpi=240, bbox_inches="tight")
print(f"Fig. 8 已保存：{figure8_path}")
plt.show()

## 6. 导出 Fig. 8 数值

CSV 的第一列是红移，后四列按论文图例顺序保存四条曲线，便于以后更换 HMFcalc 配置后逐点比较。

In [ ]:
fig8_table = np.column_stack([
    REDSHIFT,
    fig8_mono_ludlow,
    fig8_lognormal_ludlow,
    fig8_mono_prada,
    fig8_lognormal_prada,
])

fig8_csv_path = PROJECT_ROOT / "fig8_reproduction.csv"
np.savetxt(
    fig8_csv_path,
    fig8_table,
    delimiter=",",
    header=(
        "z,mono_ludlow16_gpc-3_yr-1,"
        "lognormal_ludlow16_gpc-3_yr-1,"
        "mono_prada12_gpc-3_yr-1,"
        "lognormal_prada12_gpc-3_yr-1"
    ),
    comments="",
    fmt="%.10e",
)
print(f"Fig. 8 数据已保存：{fig8_csv_path}")

## 7. 当前复现边界

论文只说明采用 HMFcalc 和 Press-Schechter，没有公开完整的宇宙学、功率谱、转移函数和质量网格配置。本项目当前明确使用 `twobodycapture.py` 中记录的 Planck18、CAMB、$\sigma_8=0.8159$、$n_s=0.9667$ 和 0.1 dex 网格。

当前结果在 $z=0$ 为 $\mathcal{O}(1)\,\mathrm{Gpc}^{-3}\,\mathrm{yr}^{-1}$，与论文图的起点量级接近；但到 $z=12$ 只有 $\mathcal{O}(10)$，低于论文图中的 $\mathcal{O}(10^2)$。此外，当前 Prada12 在高红移略高于 Ludlow16，而论文在 $z>2$ 时给出 Ludlow16 更高。这里保留真实公式输出和论文坐标，不乘经验系数，也不交换曲线标签。该差异需要从 HMFcalc 原始配置、每晕率的红移演化和 Prada12 截断细节继续定位。